# 🗂️ Notebook 2: Distributed Lock Manager — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## API

```http
POST /locks/{name}/acquire     body: { owner, ttl_ms }
  → { acquired: true, fencing_token: 17, expires_at: ... }

POST /locks/{name}/renew       body: { owner, token, ttl_ms }
  → { expires_at: ... }

POST /locks/{name}/release     body: { owner, token }
```

The **fencing token never goes down** — even across restarts of the service — so whatever
we store it in must be a monotonic counter (etcd's revision, Redis `INCR`, SQL sequence).

## Data model

| Field | Value |
|---|---|
| name | lock name |
| owner | client id |
| token | monotonically increasing integer |
| expires_at | timestamp (for lease-based eviction) |
